In [8]:
import os
import sys
import difflib
import importlib

PARSER_MODULE_NAME = 'pds_parser'
print("--- Initializing Test Environment ---")
current_working_directory = os.getcwd()
if current_working_directory not in sys.path:
    sys.path.insert(0, current_working_directory)
print(f"Current working directory: {current_working_directory}")

if PARSER_MODULE_NAME in sys.modules:
    try:
        parser_module_reloaded = importlib.reload(sys.modules[PARSER_MODULE_NAME])
        print(f"--- Module '{PARSER_MODULE_NAME}' reloaded. ---")
        from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
    except Exception as e:
        print(f"--- FAILED to reload module '{PARSER_MODULE_NAME}': {e} ---")
        from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
else:
    from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
    print(f"--- Module '{PARSER_MODULE_NAME}' imported fresh. ---")

print(f"--- Using PdsParser from: {PdsParser.__module__}.py ---")
print("-" * 80)

test_content = """
################
# WITCH EVENTS #
################

namespace = witch

#witch.1001-1999 - Guardian coverts ward
#witch.2001-2899 - Convert to witchcraft scheme

witch.1001 = { #by Mathilda Bjarnehed
    hidden = yes
    
    trigger = {
        is_witch_trigger = no
        any_relation = {
            type = guardian
            is_witch_trigger = yes
        }
    }

    immediate = {
        save_scope_as = child
        if = { # Conditional block
            limit = {
                is_ai = yes
                exists = house
                house = {
                    has_house_modifier = witch_coven
                    house_head = { is_ai = yes }
                }
                any_relation = {
                    type = guardian
                    is_ai = yes
                }
            }
            child_witch_conversion_success_effect = yes
        }
        else = {
            random_relation = { type = guardian trigger_event = witch.1002 }
        }
    }
}   

scripted_trigger witch_1002_allow_reveal_outcome_trigger = {
    exists = scope:child.liege
    scope:guardian = {
        NOT = { this = scope:child.liege }
        any_secret = {
            secret_type = secret_witch
            OR = {
                NOT = { is_known_by = scope:child }
                NOT = { is_known_by = scope:child.liege }
            }
        }
    }
}


# Standard Values
@pos_compat_high = 30
@pos_compat_medium = 15
@pos_compat_low = 5

# INTRIGUE OUTCOMES
education_intrigue_1 = {
    minimum_age = 16
    intrigue = 2
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.1
    
    ruler_designer_cost = 0
    
    culture_modifier = {
        parameter = poorly_educated_leaders_distrusted
        feudal_government_opinion = -10
    }
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_1_desc
            }
            desc = trait_education_intrigue_1_character_desc
        }
    }

    group = education_intrigue
    level = 1
}
education_intrigue_2 = {
    minimum_age = 16
    intrigue = 4
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.2
    
    ruler_designer_cost = 20
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_2_desc
            }
            desc = trait_education_intrigue_2_character_desc
        }
    }

    group = education_intrigue
    level = 2
}
"""

test_filepath = "test_events_parser_final.txt"
with open(test_filepath, "w", encoding="utf-8-sig") as f:
    f.write(test_content)
print(f"--- Created test file: {test_filepath} ---")
print("-" * 80)

parser = PdsParser()
print(f"--- Parsing '{test_filepath}'... ---")
parsed_nodes = parser.parse_file(test_filepath)

if parsed_nodes:
    print("\n--- Parsed Tree (Root Nodes Summary) ---")
    for i, node in enumerate(parsed_nodes):
        print(f"{i:03d}: {node!r}")
        if isinstance(node, PdsBlock) and node.children:
            # Limited child printing for brevity
            # pass 
            # To see more children:
            # print(f"     Children of '{node.key}' (first 5):")
            # for j, child_node in enumerate(node.children[:5]):
            #      print(f"       {j:03d}: {child_node!r}")
            # if len(node.children) > 5:
            #     print(f"       ... and {len(node.children) - 5} more children.")
            pass # Keep summary concise for now
    print("-" * 80)

    reconstructed_content = parser.to_string()
    
    # --- Detailed Diffing and Verification ---
    # Using splitlines(True) for difflib to handle newlines consistently if possible
    original_lines_for_diff = test_content.splitlines(True)
    reconstructed_lines_for_diff = reconstructed_content.splitlines(True)

    # Remove initial blank line from test_content if it exists and reconstruct doesn't make one
    if original_lines_for_diff and original_lines_for_diff[0].strip() == "":
        original_lines_for_diff = original_lines_for_diff[1:]
    if reconstructed_lines_for_diff and reconstructed_lines_for_diff[0].strip() == "":
         reconstructed_lines_for_diff = reconstructed_lines_for_diff[1:]

    diff = list(difflib.unified_diff(original_lines_for_diff, reconstructed_lines_for_diff,
                                     fromfile='Original', tofile='Reconstructed', lineterm='', n=3))
    
    # Check if the diff list contains any actual difference lines (starting with '+' or '-')
    # excluding the header lines '--- Original' and '+++ Reconstructed' and '@@ ... @@'
    actual_diff_lines = [d_line for d_line in diff if (d_line.startswith('+') or d_line.startswith('-')) and \
                                                    not d_line.startswith('---') and not d_line.startswith('+++')]

    if not actual_diff_lines:
        print("\nSUCCESS: Reconstructed content PERFECTLY matches original (or only whitespace/newline normalizations not caught by this diff).")
    else:
        print("\nWARNING: Reconstructed content differences found. Diff printed below:")
        print("Legend: '-' Original, '+' Reconstructed. Context lines are unchanged.")
        print("\n" + "="*70 + " DIFF OUTPUT " + "="*70)
        for line_diff in diff: # Print all diff lines including headers and context
            sys.stdout.write(line_diff) # Use sys.stdout.write to preserve exact line endings from diff
        print("="*70 + " END DIFF " + "="*70)

        recon_file = "reconstructed_parser_final_output.txt"
        orig_file = "original_parser_final_input.txt"
        with open(recon_file, "w", encoding="utf-8-sig") as f: f.write(reconstructed_content)
        with open(orig_file, "w", encoding="utf-8-sig") as f: f.write(test_content)
        print(f"\nFor detailed comparison, see '{orig_file}' and '{recon_file}'")
else:
    print(f"ERROR: No nodes parsed from {test_filepath}.")

# --- Test find_node (example) ---
if parsed_nodes:
    print("\n--- Testing find_node ---")
    test_block_for_find = next((n for n in parsed_nodes if isinstance(n, PdsBlock) and n.key == 'witch.1001'), None)
    if test_block_for_find:
        print(f"Searching in block: '{test_block_for_find.key}' (L{test_block_for_find.line_number})")
        path1 = 'trigger.is_witch_trigger'
        found_node1 = test_block_for_find.find_node(path1)
        print(f"Finding '{path1}': {found_node1!r}" + (f" | Value: '{found_node1.value}'" if hasattr(found_node1, 'value') else ""))
        path2 = 'trigger.any_relation.type'
        found_node2 = test_block_for_find.find_node(path2)
        print(f"Finding '{path2}': {found_node2!r}" + (f" | Value: '{found_node2.value}'" if hasattr(found_node2, 'value') else ""))
        path3 = 'immediate.if.limit.house.house_head' # Corrected path
        found_node3 = test_block_for_find.find_node(path3)
        print(f"Finding '{path3}': {found_node3!r}" + (f" | Value: '{found_node3.value}'" if hasattr(found_node3, 'value') else ""))

    target_key = 'scripted_trigger witch_1002_allow_reveal_outcome_trigger'
    scripted_trigger_node = next((n for n in parsed_nodes if hasattr(n, 'key') and n.key == target_key), None)
    if scripted_trigger_node:
        print(f"Found root node '{target_key}': {scripted_trigger_node!r}")
        if isinstance(scripted_trigger_node, PdsBlock):
            exists_node = scripted_trigger_node.find_node('exists')
            print(f"  Finding 'exists' in it: {exists_node!r}" + (f" | Value: '{exists_node.value}'" if hasattr(exists_node, 'value') else ""))
print("\n" + "="*80)
print("Parser Test Run Complete.")
print("="*80)

--- Initializing Test Environment ---
Current working directory: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update
--- Module 'pds_parser' reloaded. ---
--- Using PdsParser from: pds_parser.py ---
--------------------------------------------------------------------------------
--- Created test file: test_events_parser_final.txt ---
--------------------------------------------------------------------------------
--- Parsing 'test_events_parser_final.txt'... ---

--- Parsed Tree (Root Nodes Summary) ---
000: <PdsBlankLine L1 C1 Key='Blank Line'>
001: <PdsComment L2 C1 Key='Comment: '###############...''>
002: <PdsComment L3 C1 Key='Comment: 'WITCH EVENTS #...''>
003: <PdsComment L4 C1 Key='Comment: '###############...''>
004: <PdsBlankLine L5 C1 Key='Blank Line'>
005: <PdsKeyValuePair L6 C1 Key='namespace'>
006: <PdsBlankLine L7 C1 Key='Blank Line'>
007: <PdsComment L8 C1 Key='Comment: 'witch.1001-1999 - Guardian cov...''>
008: <PdsComment L9 C1 Key='Comment: 'witch.2001-2899 - Convert to w...

In [ ]:
## LEXER TEST  ##

import os
import sys
import importlib # Added for attempting to reload the module

# --- Configuration ---
LEXER_MODULE_NAME = 'pds_lexer' # Ensure your lexer file is pds_lexer.py

# --- Attempt to reload the lexer module ---
# This is to help ensure the latest version is used, but restarting the kernel is more reliable.
if LEXER_MODULE_NAME in sys.modules:
    try:
        lexer_module = importlib.reload(sys.modules[LEXER_MODULE_NAME])
        print(f"--- Module '{LEXER_MODULE_NAME}' reloaded successfully. ---")
    except Exception as e:
        print(f"--- FAILED to reload module '{LEXER_MODULE_NAME}': {e} ---")
else:
    print(f"--- Module '{LEXER_MODULE_NAME}' not yet imported, will import fresh. ---")

# --- Path setup ---
# Ensure pds_lexer.py is in the Python path
current_working_directory = os.getcwd()
if current_working_directory not in sys.path:
    sys.path.insert(0, current_working_directory)
print(f"--- Current working directory: {current_working_directory} ---")
print(f"--- Python sys.path (first few entries): {sys.path[:3]} ---")

# --- Import Lexer (after potential reload) ---
try:
    from pds_lexer import PdsLexer, PdsToken
    print("--- PdsLexer and PdsToken imported successfully. ---")
except ImportError as e:
    print(f"--- FATAL: Could not import PdsLexer or PdsToken: {e} ---")
    print("--- Please ensure 'pds_lexer.py' is in the same directory or Python path and contains these classes. ---")
    # Stop further execution if import fails
    raise

# --- Test Content ---
test_content = """
################
# WITCH EVENTS #
################

namespace = witch

#witch.1001-1999 - Guardian coverts ward
#witch.2001-2899 - Convert to witchcraft scheme

witch.1001 = { #by Mathilda Bjarnehed
    hidden = yes
    
    trigger = {
        is_witch_trigger = no
        any_relation = {
            type = guardian
            is_witch_trigger = yes
        }
    }

    immediate = {
        save_scope_as = child
        if = { # Conditional block
            limit = {
                is_ai = yes
                exists = house
                house = {
                    has_house_modifier = witch_coven
                    house_head = { is_ai = yes }
                }
                any_relation = {
                    type = guardian
                    is_ai = yes
                }
            }
            child_witch_conversion_success_effect = yes
        }
        else = {
            random_relation = { type = guardian trigger_event = witch.1002 }
        }
    }
}   

scripted_trigger witch_1002_allow_reveal_outcome_trigger = {
    exists = scope:child.liege
    scope:guardian = {
        NOT = { this = scope:child.liege }
        any_secret = {
            secret_type = secret_witch
            OR = {
                NOT = { is_known_by = scope:child }
                NOT = { is_known_by = scope:child.liege }
            }
        }
    }
}


# Standard Values
@pos_compat_high = 30
@pos_compat_medium = 15
@pos_compat_low = 5

# INTRIGUE OUTCOMES
education_intrigue_1 = {
    minimum_age = 16
    intrigue = 2
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.1
    
    ruler_designer_cost = 0
    
    culture_modifier = {
        parameter = poorly_educated_leaders_distrusted
        feudal_government_opinion = -10
    }
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_1_desc
            }
            desc = trait_education_intrigue_1_character_desc
        }
    }

    group = education_intrigue
    level = 1
}
education_intrigue_2 = {
    minimum_age = 16
    intrigue = 4
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.2
    
    ruler_designer_cost = 20
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_2_desc
            }
            desc = trait_education_intrigue_2_character_desc
        }
    }

    group = education_intrigue
    level = 2
}
"""

# --- Helper function to inspect characters ---
def inspect_text_at_location(text, target_line, target_col, window=20):
    """Inspects characters around a given line and column (1-indexed)."""
    print(f"\n--- Inspecting text_content around L{target_line}, C{target_col} ---")
    lines = text.splitlines(True) # Keep line endings
    if not (0 < target_line <= len(lines)):
        print(f"ERROR: Target line {target_line} is out of bounds (1-{len(lines)}).")
        return

    line_content = lines[target_line - 1]
    
    # Adjust column to be 0-indexed for string slicing
    char_index = target_col - 1

    if not (0 <= char_index < len(line_content)):
        print(f"ERROR: Target column {target_col} is out of bounds for line {target_line} (length {len(line_content)}).")
        print(f"Line content: '{line_content.rstrip()}'")
        return

    actual_char = line_content[char_index]
    print(f"Character at L{target_line}, C{target_col}: '{actual_char}' (ord: {ord(actual_char)}, hex: {hex(ord(actual_char))})")

    start = max(0, char_index - window // 2)
    end = min(len(line_content), char_index + window // 2 + 1)
    
    context_before = line_content[start:char_index]
    context_after = line_content[char_index+1:end]
    
    print(f"Context: '{context_before}<HERE>{actual_char}<HERE>{context_after.rstrip()}'")
    print(f"Line (raw): {repr(line_content)}")


# --- Lexer Test Execution ---
print("\n--- Starting Lexer Test ---")

# Inspect the suspected character location IN THE ORIGINAL test_content string
# This helps verify if the input string itself has an anomaly.
# The error is at line 11, col 16.
# Note: Line counting for `test_content` string literal starts after the initial triple quotes.
# The first actual content line "################" is line 2 if we count the initial blank line.
# Let's find the absolute character position for L11, C16 of the *parsed content*.
# The lexer's line counting starts at 1.
# The problematic line in the content block: "witch.1001 = { #by Mathilda Bjarnehed"

# To find the absolute position for `inspect_text_at_location`, we need to be careful.
# Let's assume the lexer's line 11 refers to the 11th non-empty or significant line.
# The content starts with a newline.
# 1: (empty)
# 2: ################
# 3: # WITCH EVENTS #
# 4: ################
# 5: (empty)
# 6: namespace = witch
# 7: (empty)
# 8: #witch.1001-1999 - Guardian coverts ward
# 9: #witch.2001-2899 - Convert to witchcraft scheme
#10: (empty)
#11: witch.1001 = { #by Mathilda Bjarnehed  <-- This is the line
inspect_text_at_location(test_content, 11, 16)


lexer = PdsLexer(test_content)
tokens = [] # Initialize tokens list

try:
    tokens = lexer.tokenize()
    print(f"--- Lexing Complete. Found {len(tokens)} tokens. ---")
    
    if tokens: # Check if tokens list is not empty
        print("\n--- First 20 Tokens: ---")
        for i, token in enumerate(tokens[:20]):
            print(f"{i:03d}: {token}")
        
        print("\n--- Last 20 Tokens (including EOF): ---")
        for i, token in enumerate(tokens[-20:], start=max(0, len(tokens)-20)):
            print(f"{i:03d}: {token}")

        # Example: Find tokens around a specific line where the error might be
        error_line = 11
        print(f"\n--- Tokens around Line {error_line} (and +/- 1 line): ---")
        for token in tokens:
            if error_line -1 <= token.line <= error_line + 1:
                print(token)
    else:
        print("--- No tokens were generated. ---")

except ValueError as e:
    print(f"LEXER ERROR: {e}")
    print("\n--- Lexer state at point of error (if accessible, depends on lexer structure): ---")
    print(f"Lexer Position: {getattr(lexer, 'pos', 'N/A')}")
    print(f"Lexer Line: {getattr(lexer, 'line', 'N/A')}")
    print(f"Lexer Column: {getattr(lexer, 'column', 'N/A')}")
    if hasattr(lexer, 'pos') and hasattr(lexer, 'text'):
        error_pos = lexer.pos
        text_context_start = max(0, error_pos - 30)
        text_context_end = min(len(lexer.text), error_pos + 30)
        print(f"Text context around error (pos {error_pos}):")
        print(f"...'{lexer.text[text_context_start:error_pos]}<ERROR_HERE>{lexer.text[error_pos:text_context_end]}'...")
except Exception as e_general:
    print(f"AN UNEXPECTED ERROR OCCURRED: {e_general}")
    import traceback
    traceback.print_exc()

print("\n--- Lexer Test Complete ---")

--- Module 'pds_lexer' not yet imported, will import fresh. ---
--- Current working directory: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update ---
--- Python sys.path (first few entries): ['c:\\Users\\Galaxy\\LEVI\\jupyter\\ck3_mod_update', 'c:\\Users\\Galaxy\\miniconda3\\python312.zip', 'c:\\Users\\Galaxy\\miniconda3\\DLLs'] ---
--- PdsLexer and PdsToken imported successfully. ---

--- Starting Lexer Test ---

--- Inspecting text_content around L11, C16 ---
Character at L11, C16: '#' (ord: 35, hex: 0x23)
Context: '.1001 = { <HERE>#<HERE>by Mathild'
Line (raw): 'witch.1001 = { #by Mathilda Bjarnehed\n'
--- DEBUGGING BLOCK IN PdsLexer ACTIVATED ---
LEXER_DEBUG: Current char: '#' (Unicode ord: 35), Line: 11, Column: 16, Pos: 177
LEXER_DEBUG: Text context: 'witch.1001 = { <HERE>#by Mathilda Bja'
LEXER_DEBUG: Slice for COMMENT match (first 30 chars): '#by Mathilda Bjarnehed
    hid'
LEXER_DEBUG: Direct test of COMMENT pattern SUCCEEDED. Group(0): '#by Mathilda Bjarnehed'
--- END LEXER DEBUGGIN

In [1]:
# Jupyter Notebook Cell

import os
import sys

# Ensure pds_parser.py and pds_differ.py are in the same directory
# Restart your Jupyter Notebook kernel before running this cell if files have changed!
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsBlankLine
from pds_differ import PdsDiffer, PdsChange

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Define Test File Contents ---

# Test Case 1: Simple changes, additions, modifications
# Original state:
old_vanilla_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 10
    param_b2 = text_b2
}
key_c = value_c_old
"""

# Mod's changes: added key_x, modified key_c, modified block_b (param_b1 changed)
mod_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 20 # Mod changed this
    param_b2 = text_b2
    param_b3 = new_param_by_mod # Mod added this
}
key_c = value_c_mod # Mod changed this
key_x = value_x_by_mod # Mod added this
"""

# New Vanilla's changes: modified key_c, added key_y, modified block_b (param_b2 changed)
new_vanilla_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 10
    param_b2 = text_b2_new # NV changed this
    param_b4 = new_param_by_nv # NV added this
}
key_c = value_c_nv # NV changed this (CONFLICT with mod)
key_y = value_y_by_nv # NV added this
"""

# Test Case 2: Deletions, Block type change (simplified)
# old_vanilla: has a block, mod deletes it, new vanilla changes it to a simple KV
old_vanilla_content_2 = """
file_version = 1.0
trait_block = {
    attr_a = 1
    attr_b = 2
}
event_id = 123
"""

# Mod's changes: trait_block is deleted, event_id modified
mod_content_2 = """
file_version = 1.0
event_id = 456_mod # Mod changed this
"""

# New Vanilla's changes: trait_block becomes a KV, event_id changed by NV
new_vanilla_content_2 = """
file_version = 1.1 # NV updated
trait_block = "simplified" # NV changed block to KV
event_id = 789_nv # NV changed this (CONFLICT with mod)
"""


# --- Helper Function to Run Diff ---
def run_and_print_diff(test_name, old_content, mod_content, new_content):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    
    # Write to temporary files for parsing
    with open("temp_old.txt", "w", encoding="utf-8-sig") as f: f.write(old_content)
    with open("temp_mod.txt", "w", encoding="utf-8-sig") as f: f.write(mod_content)
    with open("temp_new.txt", "w", encoding="utf-8-sig") as f: f.write(new_content)

    # Parse files
    parser = PdsParser()
    old_nodes = parser.parse_file("temp_old.txt")
    mod_nodes = parser.parse_file("temp_mod.txt")
    new_nodes = parser.parse_file("temp_new.txt")

    if not old_nodes or not mod_nodes or not new_nodes:
        print(f"ERROR: Failed to parse one or more files for test '{test_name}'.")
        return

    # Run the differ
    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)

    print(f"\n--- Detected Changes for '{test_name}' ({len(changes)} changes) ---")
    for change in changes:
        print(change)
    print(f"{'='*20} END DIFF TEST: {test_name} {'='*20}\n")

# --- Run Tests ---
run_and_print_diff("Test Case 1: Simple Changes", old_vanilla_content_1, mod_content_1, new_vanilla_content_1)
run_and_print_diff("Test Case 2: Deletions and Type Changes", old_vanilla_content_2, mod_content_2, new_vanilla_content_2)

# Clean up temp files (optional, but good practice)
try:
    os.remove("temp_old.txt")
    os.remove("temp_mod.txt")
    os.remove("temp_new.txt")
except OSError:
    pass # File might not exist if parsing failed

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------

==================== RUNNING DIFF TEST: Test Case 1: Simple Changes ====================

--- Detected Changes for 'Test Case 1: Simple Changes' (7 changes) ---
PdsChange(Type='MOD_MODIFIED                       ', Path='block_b.param_b1', Parent='block_b', 
           Nodes=[O:param_b1='10', M:param_b1='20', N:param_b1='10'])
PdsChange(Type='VANILLA_MODIFIED                   ', Path='block_b.param_b2', Parent='block_b', 
           Nodes=[O:param_b2='text_b2', M:param_b2='text_b2', N:param_b2='text_b2_new'])
PdsChange(Type='MOD_ADDED_VANILLA_IGNORES          ', Path='block_b.param_b3', Parent='block_b', 
           Nodes=[O:ABSENT, M:param_b3='new_param_by_mod', N:ABSENT])
PdsChange(Type='VANILLA_ADDED_MOD_IGNORES          ', Path='block_b.param_b4', Parent='block_b', 
           Nodes